# Fine-tune GLM-OCR for Vietnamese Diacritical Marks (v2)

Follows the official guide: [examples/finetune/README.md](https://github.com/zai-org/GLM-OCR/blob/main/examples/finetune/README.md)

**Dataset format:** ShareGPT with `messages`/`role`/`content` + `images` fields.

**Workflow:**
1. Chạy các bước 1→6 (setup, chỉ 1 lần)
2. Bước 7a: Train epoch đầu tiên
3. Bước 8: Merge & test kết quả
4. Nếu chưa ổn → chạy bước 7b (thêm 1 epoch) → quay lại bước 8 test
5. Lặp cho đến khi hài lòng → bước 10 save

**Requirements:**
- GPU: T4 (16GB) or better
- Upload `vietnamese_ocr.zip` to Google Drive `My Drive` root
  - Structure: `vietnamese_ocr/vietnamese_ocr.json` + `vietnamese_ocr/images/txt_*.png`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Restore LoRA checkpoint from Drive (neu bi disconnect)
import os
drive_ckpt = "/content/drive/My Drive/glm-ocr-vn-checkpoints"
local_ckpt = "/content/glm-ocr-lora-sft"

if os.path.exists(drive_ckpt):
    os.makedirs(local_ckpt, exist_ok=True)
    !cp -r "{drive_ckpt}"/* "{local_ckpt}"/
    ckpts = [d for d in os.listdir(local_ckpt) if d.startswith("checkpoint-")]
    if ckpts:
        print(f"Restored {len(ckpts)} checkpoint(s):")
        for c in sorted(ckpts):
            print(f"  {c}")
    else:
        print("No checkpoints found in Drive. Will train from scratch.")
else:
    print("No saved checkpoints on Drive. Will train from scratch.")

In [ ]:
# Extract dataset
!cp "/content/drive/My Drive/vietnamese_ocr.zip" /content/
!cd /content && unzip -q -o vietnamese_ocr.zip
!echo "=== Structure ==="
!ls /content/vietnamese_ocr/
!echo "Images:" 0
!echo "JSON:" 

In [ ]:
# Verify dataset format: must have messages/role/content
import json, os

with open("/content/vietnamese_ocr/vietnamese_ocr.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")
print(f"\nSample 0:")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

# Verify format
sample = data[0]
assert "messages" in sample, "ERROR: missing 'messages' key"
assert "images" in sample, "ERROR: missing 'images' key"
assert sample["messages"][0]["role"] == "user", "ERROR: first message must be user"
assert sample["messages"][1]["role"] == "assistant", "ERROR: second message must be assistant"
assert "<image>" in sample["messages"][0]["content"], "ERROR: user message must contain <image>"
print("\n✓ Dataset format is correct!")

# Check image exists
img_path = os.path.join("/content/vietnamese_ocr", sample["images"][0])
print(f"\nImage path: {img_path}")
print(f"Image exists: {os.path.exists(img_path)}")

## 3. Install LLaMA-Factory

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
!pip install -e ".[torch,metrics]" 2>&1 | tail -5

In [ ]:
# Pin transformers to 5.6.0 — compatible with both LLaMA-Factory (>=4.55, <=5.6.0) and GLM-OCR (>=5.3.0)
!pip install transformers==5.6.0 2>&1 | tail -3

In [ ]:
!llamafactory-cli version

## 4. Download GLM-OCR Model

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    "zai-org/GLM-OCR",
    local_dir="/content/GLM-OCR",
    local_dir_use_symlinks=False,
)
print(f"Model downloaded to: {model_dir}")

## 5. Prepare Dataset for LLaMA-Factory

Copy dataset into `LLaMA-Factory/data/` (image paths are relative to this directory).

In [ ]:
# Copy dataset into LLaMA-Factory/data/
!cp /content/vietnamese_ocr/vietnamese_ocr.json /content/LLaMA-Factory/data/
!test -f /content/vietnamese_ocr/vietnamese_ocr_test.json && cp /content/vietnamese_ocr/vietnamese_ocr_test.json /content/LLaMA-Factory/data/ || true
!cp -r /content/vietnamese_ocr/images /content/LLaMA-Factory/data/images
!echo "Images:" 0
!echo "JSON:" 

In [ ]:
# Verify image path resolution
import json, os

with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
    data = json.load(f)

sample = data[0]
img_rel = sample["images"][0]
img_abs = os.path.join("/content/LLaMA-Factory/data", img_rel)
print(f"Image relative path: {img_rel}")
print(f"Image absolute path: {img_abs}")
print(f"Exists: {os.path.exists(img_abs)}")

# Check all images exist
missing = 0
for item in data:
    for img in item["images"]:
        if not os.path.exists(os.path.join("/content/LLaMA-Factory/data", img)):
            missing += 1
print(f"\nMissing images: {missing}/{len(data)}")

In [ ]:
# Register dataset in dataset_info.json
import json

ds_info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(ds_info_path, "r") as f:
    info = json.load(f)

info["vietnamese_ocr"] = {
    "file_name": "vietnamese_ocr.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "messages",
        "images": "images"
    },
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant"
    }
}

info["vietnamese_ocr_test"] = {
    "file_name": "vietnamese_ocr_test.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "messages",
        "images": "images"
    },
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant"
    }
}

with open(ds_info_path, "w") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("✓ Dataset registered in dataset_info.json")
print(json.dumps(info["vietnamese_ocr"], indent=2))

## 6. Write Training Config

In [ ]:
yaml_content = """
### model
model_name_or_path: /content/GLM-OCR
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### dataset
dataset: vietnamese_ocr
template: glm_ocr
cutoff_len: 2048
preprocessing_num_workers: 8
dataloader_num_workers: 2
val_size: 0.0
per_device_eval_batch_size: 1

### output
output_dir: /content/glm-ocr-lora-sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: 1
lr_scheduler_type: constant_with_warmup
warmup_ratio: 0.05
fp16: true
"""

with open("/content/glm_ocr_vn_lora_sft.yaml", "w") as f:
    f.write(yaml_content)

print("Config written to /content/glm_ocr_vn_lora_sft.yaml")
print(yaml_content)

---

## 7. Train (nhan lai de train them epoch)

Moi lan chay = them 1 epoch. Tu detect checkpoint cu de resume.
- **Lan dau:** train epoch 1 tu dau
- **Lan sau:** resume tu checkpoint cu, train them 1 epoch

Mat khoang 2 phut/epoch tren T4. Checkpoint duoc tu luu len Drive sau khi train xong.

In [ ]:
# Reset: xoa toan bo checkpoint de train tu dau
!rm -rf /content/glm-ocr-lora-sft/checkpoint-*
!rm -rf "/content/drive/My Drive/glm-ocr-vn-checkpoints"
print("Deleted all checkpoints. Next train will start from epoch 1.")

In [ ]:
import os, json

# Detect checkpoint
ckpt_dir = "/content/glm-ocr-lora-sft"
checkpoints = sorted([d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")]) if os.path.exists(ckpt_dir) else []

if checkpoints:
    last_ckpt = os.path.join(ckpt_dir, checkpoints[-1])
    step = int(checkpoints[-1].split("-")[1])
    with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
        _ds = json.load(f)
    steps_per_epoch = int(len(_ds) * 0.9) // (4 * 4)
    next_epoch = step // steps_per_epoch + 1
    total_epochs = next_epoch + 1
    print(f"Resuming from {checkpoints[-1]} (epoch {next_epoch}) -> training to epoch {total_epochs}")
else:
    last_ckpt = None
    total_epochs = 1
    print("No checkpoint found -> training epoch 1")

# Write YAML
yaml_content = f"""
### model
model_name_or_path: /content/GLM-OCR
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### dataset
dataset: vietnamese_ocr
template: glm_ocr
cutoff_len: 2048
preprocessing_num_workers: 8
dataloader_num_workers: 2
val_size: 0.0
per_device_eval_batch_size: 1

### output
output_dir: /content/glm-ocr-lora-sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: false
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: {total_epochs}
lr_scheduler_type: constant_with_warmup
warmup_ratio: 0.05
fp16: true
"""
if last_ckpt:
    yaml_content += f"resume_from_checkpoint: {last_ckpt}\n"

with open("/content/glm_ocr_vn_lora_sft.yaml", "w") as f:
    f.write(yaml_content)

os.environ["DISABLE_VERSION_CHECK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0
!llamafactory-cli train /content/glm_ocr_vn_lora_sft.yaml

In [ ]:
# Auto-save checkpoint to Drive
!mkdir -p "/content/drive/My Drive/glm-ocr-vn-checkpoints"
!cp -r /content/glm-ocr-lora-sft/checkpoint-* "/content/drive/My Drive/glm-ocr-vn-checkpoints/"
print("Saved checkpoints to Drive")
!ls "/content/drive/My Drive/glm-ocr-vn-checkpoints/"

## 8. Merge & Quick Test

Sau mỗi epoch, merge LoRA weights và test nhanh trên 1 ảnh.

In [ ]:
# Merge LoRA weights
!llamafactory-cli export \
  --model_name_or_path /content/GLM-OCR \
  --adapter_name_or_path /content/glm-ocr-lora-sft \
  --template glm_ocr \
  --export_dir /content/glm-ocr-vn-merged \
  --trust_remote_code true

In [ ]:
# Install eval dependency
!pip install editdistance -q

import editdistance, json, os, random, glob
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from IPython.display import display, HTML

# ── Config ──
EVAL_N_SAMPLES = 100
RESULTS_FILE = "/content/drive/My Drive/glm-ocr-vn-eval-history.json"
TEST_JSON = "/content/vietnamese_ocr/vietnamese_ocr_test.json"
MERGED_DIR = "/content/glm-ocr-vn-merged"

# ── Diacritic groups ──
DIACRITIC_GROUPS = {
    "ă (ắằẳẵặ)": set("ăắằẳẵặ"),
    "â (ấầẩẫậ)": set("âấầẩẫậ"),
    "ê (ếềểễệ)": set("êếềểễệ"),
    "ô (ốồổỗộ)": set("ôốồổỗộ"),
    "ơ (ớờởỡợ)": set("ơớờởỡợ"),
    "ư (ứừửữự)": set("ưứừửữự"),
    "đ": set("đĐ"),
}
ALL_DIACRITICS = set().union(*DIACRITIC_GROUPS.values())

# ── Load model ──
print("Loading merged model...")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MERGED_DIR, trust_remote_code=True, torch_dtype="auto", device_map="auto"
)

# ── Load test set ──
if not os.path.exists(TEST_JSON):
    # Fallback: use full dataset, take last 10%
    print("⚠️ No separate test set found. Using full dataset with val split.")
    TEST_JSON = "/content/vietnamese_ocr/vietnamese_ocr.json"

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)
if EVAL_N_SAMPLES < len(test_data):
    random.seed(42)
    test_data = random.sample(test_data, EVAL_N_SAMPLES)
print(f"Evaluating on {len(test_data)} test samples...")

# ── Run evaluation ──
stats = {
    "cer_total": 0, "char_total": 0,
    "wer_total": 0, "word_total": 0,
    "d_correct": {g: 0 for g in DIACRITIC_GROUPS},
    "d_total": {g: 0 for g in DIACRITIC_GROUPS},
    "exact_match": 0,
}

for i, item in enumerate(test_data):
    gt = item["messages"][1]["content"]
    img_path = os.path.join("/content/vietnamese_ocr", item["images"][0])

    messages = [{"role": "user", "content": [
        {"type": "image", "url": img_path},
        {"type": "text", "text": "Text Recognition:"},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    inputs.pop("token_type_ids", None)

    generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    pred = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Exact match
    if pred == gt:
        stats["exact_match"] += 1

    # CER
    stats["cer_total"] += editdistance.eval(pred, gt)
    stats["char_total"] += max(len(gt), 1)

    # WER
    gt_words = gt.split()
    pred_words = pred.split()
    stats["wer_total"] += editdistance.eval(pred_words, gt_words)
    stats["word_total"] += max(len(gt_words), 1)

    # Diacritic accuracy per group
    for group_name, chars in DIACRITIC_GROUPS.items():
        for c_gt, c_pred in zip(gt, pred):
            if c_gt in chars:
                stats["d_total"][group_name] += 1
                if c_gt == c_pred:
                    stats["d_correct"][group_name] += 1

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(test_data)} done")

# ── Compute metrics ──
cer = stats["cer_total"] / max(stats["char_total"], 1)
wer = stats["wer_total"] / max(stats["word_total"], 1)
em = stats["exact_match"] / len(test_data) * 100
total_dc = sum(stats["d_correct"].values())
total_dt = sum(stats["d_total"].values())
overall_dacc = total_dc / max(total_dt, 1) * 100

# ── Load/save history ──
history = []
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        history = json.load(f)

current_epoch = len(history) + 1
epoch_result = {"epoch": current_epoch, "n_samples": len(test_data),
                "CER": round(cer, 4), "WER": round(wer, 4),
                "Exact_Match%": round(em, 1), "Diacritic_Acc%": round(overall_dacc, 1)}
for g in DIACRITIC_GROUPS:
    t, c = stats["d_total"][g], stats["d_correct"][g]
    epoch_result[g] = round(c / max(t, 1) * 100, 1) if t > 0 else None

history.append(epoch_result)
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
with open(RESULTS_FILE, "w") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

# ── Display current results ──
print(f"\n{'='*78}")
print(f"  📊 EPOCH {current_epoch} — {len(test_data)} test samples")
print(f"{'='*78}")
print(f"  CER (Character Error Rate):  {cer:.4f}  ← càng thấp càng tốt, 0.0 = hoàn hảo")
print(f"  WER (Word Error Rate):       {wer:.4f}  ← càng thấp càng tốt, 0.0 = hoàn hảo")
print(f"  Exact Match:                 {em:.1f}%   ← % sample đoán đúng 100% từng ký tự")
print(f"  Diacritic Acc:               {overall_dacc:.1f}%   ← % dấu TV đoán đúng ({total_dc}/{total_dt})")
print(f"\n  Chi tiết theo từng nhóm nguyên âm (càng cao càng tốt):")
for g in DIACRITIC_GROUPS:
    t, c = stats["d_total"][g], stats["d_correct"][g]
    acc = c / max(t, 1) * 100 if t > 0 else 0
    bar = "█" * int(acc / 5) + "░" * (20 - int(acc / 5))
    print(f"    {g:20s} {acc:5.1f}%  {bar}  ({c}/{t})")

# ── Progress table across epochs ──
if len(history) > 0:
    print(f"\n{'='*78}")
    print(f"  📈 PROGRESS ACROSS ALL EPOCHS")
    print(f"{'='*78}")
    print(f"  Chú thích cột:")
    print(f"    Ep  = Epoch thứ mấy")
    print(f"    CER = Character Error Rate (càng THẤP càng tốt)")
    print(f"    WER = Word Error Rate (càng THẤP càng tốt)")
    print(f"    EM% = Exact Match % (càng CAO càng tốt)")
    print(f"    DA% = Diacritic Accuracy % — % dấu TV đoán đúng (càng CAO càng tốt)")
    print(f"    ăâêôơưđ = Accuracy riêng từng nhóm nguyên âm kép (càng CAO càng tốt)")
    print()
    
    group_keys = list(DIACRITIC_GROUPS.keys())
    short_names = ["ă", "â", "ê", "ô", "ơ", "ư", "đ"]
    headers = ["Ep", "CER↓", "WER↓", "EM%↑", "DA%↑",] + short_names
    rows = []
    for h in history:
        row = [str(h.get("epoch", "?")),
               f"{h.get('CER',0):.4f}",
               f"{h.get('WER',0):.4f}",
               f"{h.get('Exact_Match%',0):.1f}",
               f"{h.get('Diacritic_Acc%',0):.1f}"]
        for g in group_keys:
            v = h.get(g)
            row.append(f"{v:.1f}" if v is not None else "N/A")
        rows.append(row)
    
    cw = [max(len(r[i]) for r in [headers] + rows) for i in range(len(headers))]
    sep = "+".join(["-" * (w + 2) for w in cw])
    def fmt_row(r):
        return "|" + "|".join(f" {r[i]:<{cw[i]}} " for i in range(len(headers))) + "|"
    
    print(f"  +{sep}+")
    print(f"  {fmt_row(headers)}")
    print(f"  +{sep}+")
    for r in rows:
        print(f"  {fmt_row(r)}")
    print(f"  +{sep}+")

    # Delta vs previous epoch
    if len(history) >= 2:
        prev, curr = history[-2], history[-1]
        print(f"\n  📊 So sánh epoch {prev.get('epoch','?')} → {curr.get('epoch','?')}:")
        print(f"    🟢 = cải thiện  |  🔴 = giảm  |  ⚪ = không đổi")
        display_names = {"CER": "CER (lỗi ký tự)↓", "WER": "WER (lỗi từ)↓",
                         "Exact_Match%": "Exact Match%↑", "Diacritic_Acc%": "Diacritic Acc%↑"}
        for metric in ["CER", "WER", "Exact_Match%", "Diacritic_Acc%"]:
            p, c = prev.get(metric, 0), curr.get(metric, 0)
            diff = c - p
            better = (metric in ["CER", "WER"] and diff < 0) or (metric not in ["CER", "WER"] and diff > 0)
            icon = "🟢" if better else "🔴" if diff != 0 else "⚪"
            name = display_names.get(metric, metric)
            print(f"    {icon} {name:25s} {p} → {c}  ({'+' if diff>=0 else ''}{diff:.2f})")

print(f"\n✅ Results saved to {RESULTS_FILE}")


**📖 Hướng dẫn đọc bảng kết quả:**

| Cột | Ý nghĩa | Tốt khi |
|-----|---------|----------|
| **CER** | Character Error Rate — tỷ lệ ký tự sai | **Thấp** (0.0 = hoàn hảo) |
| **WER** | Word Error Rate — tỷ lệ từ sai | **Thấp** (0.0 = hoàn hảo) |
| **EM%** | Exact Match — % sample đoán đúng 100% | **Cao** |
| **DA%** | Diacritic Accuracy — % dấu TV đoán đúng | **Cao** (≥95% = tốt) |
| **ă â ê ô ơ ư đ** | Accuracy riêng từng nhóm nguyên âm kép | **Cao** (nhóm nào thấp cần thêm data) |

- 🟢 = cải thiện so epoch trước | 🔴 = giảm | ⚪ = không đổi
- Nếu DA% ≥ 95% → chuyển xuống **bước 10** save model
- Nếu DA% < 95% hoặc có nhóm < 80% → quay **bước 7** train thêm epoch → chạy lại eval
- Nếu epoch mới toàn 🔴 → **overfitting**, dùng checkpoint epoch trước


---

## 10. Save to Google Drive

In [ ]:
!mkdir -p "/content/drive/My Drive/glm-ocr-vn"
!cp -r /content/glm-ocr-vn-merged/* "/content/drive/My Drive/glm-ocr-vn/"
print("✓ Model saved to Drive: My Drive/glm-ocr-vn/")